# 05 · BFTS demo — tree grows across 4 stages

In [ ]:

import sys, pathlib
ROOT = pathlib.Path().resolve().parent if pathlib.Path('notebooks').exists() else pathlib.Path().resolve()
sys.path.insert(0, str(ROOT))


In [ ]:
import random
from config import BFTSConfig
from src.optimizer.bfts_loop import BFTSLoop
from src.optimizer.tree_node import Stage
from src.visualization import plot_tree, plot_trajectory, plot_ablation_bar

rng = random.Random(0)
def run_fn(cfg, docs, queries):
    return [{'question':q['question'],'answer':'ok','retrieved_contexts':[]} for q in queries]
def eval_fn(results):
    return {'ragas_score': 0.3 + rng.random()*0.6}

loop = BFTSLoop(documents=['d'], testset=[{'question':'q'}], run_fn=run_fn, eval_fn=eval_fn,
                bfts_config=BFTSConfig(num_seeds=3, max_steps=15,
                    stage_budgets={Stage.PRELIMINARY:3, Stage.BASELINE:4, Stage.EXPLORATION:6, Stage.ABLATION:3}))

events = []
for ev in loop.run_iter():
    events.append(ev)
    if ev['type'] in ('stage_transition', 'ablation_complete', 'search_complete'):
        print(ev['type'], '->', ev.get('current_stage') or ev.get('summary', {}).get('best_score'))


In [ ]:
summary = loop._final_summary()
print('best score:', summary['best_score'])
print('tree:', summary['tree_summary'])
for t in summary['stage_transitions']:
    print(' ', t)


In [ ]:
plot_tree(loop.get_tree_visualization_data())


In [ ]:
plot_trajectory(summary['trajectory'], summary['stage_transitions'])


In [ ]:
plot_ablation_bar(summary['ablation_report'])
